In [ ]:
import os, sys
import numpy as np
from pathlib import Path
from SpaceBalls.paths import CONFIG_DIR
import SpaceBalls.radiation_settings as rad_settings
sys.path.insert(0, str(CONFIG_DIR.parent))  # parent of 'config'

base_dir = os.path.join(CONFIG_DIR, 'earth/albedo_and_thermal')

p = Path(base_dir)
subdirs = [d for d in p.iterdir() if d.is_dir()]

for dir in subdirs:

    ps = Path(dir)
    
    numpy_dir = os.path.join(base_dir, ps.stem, 'numpy_format')
    if not(os.path.exists(numpy_dir)):
        os.makedirs(numpy_dir)

    files = [f.name for f in ps.iterdir() if f.is_file()]
    all_dates = np.unique([fname.split('_')[1] for fname in files])
    all_Nmax = np.unique([fname.split('_')[2] for fname in files])
    assert(len(all_Nmax)==1)
    Nmax = int(all_Nmax[0].removeprefix('Nmax').removesuffix('.py'))

    for type_str in ["Albedo", "Thermal"]:
        for date in all_dates:
            
            file_name = os.path.join(numpy_dir, type_str + '_' + date)
            if not(os.path.exists(file_name)):

                C_mat, S_mat = rad_settings.get_CS_mats(type_str, date, Nmax, Nmax,
                                                        subfolder=dir.stem + '.',
                                                        name_tail=date + '_Nmax' + str(Nmax))
                
                CS_mat_numpy = rad_settings.convert_sh_maps_monte_to_numpy(C_mat, S_mat)
                np.save(file_name, CS_mat_numpy)



In [5]:
# r sun file organizer
import sys, os
from SpaceBalls.paths import MEDIA_DIR, OUTPUT_DIR
import numpy as np
from SpaceBalls.radiation_fluxes_preprocessing import mid_day_jd_array_from_jd_interval
import SpaceBalls.radiation_settings as rad_settings
from SpaceBalls.sph_meshing import progress_bar

rad_config = rad_settings.radiation_settings_from_EEI_truth_name("EEI_truth_1") # add any other here
mid_day_jd_array = mid_day_jd_array_from_jd_interval(rad_config["jd_interval"])
dir_0 = os.path.join(MEDIA_DIR, 'solar_ephemerides', rad_config["ephemerides"])

np.save(os.path.join(dir_0, 'daily_files', 'jd_mid_day_array'), mid_day_jd_array)


full_r_sun = np.loadtxt(os.path.join(dir_0, 'r_Sun_ECEF_hist.txt'))
full_jd_r_sun = np.loadtxt(os.path.join(dir_0, 'jd_r_Sun_ECEF_hist.txt'))

full_R_ECEF_to_SunFrame_hist = np.loadtxt(os.path.join(dir_0, 'R_ECEF_to_SunFrame_hist.txt'))
full_R_ECEF_to_SunFrame_hist = full_R_ECEF_to_SunFrame_hist.reshape(full_R_ECEF_to_SunFrame_hist.shape[0], 
                                              full_R_ECEF_to_SunFrame_hist.shape[1]//3,
                                              3)
rull_jd_R_hist = np.loadtxt(os.path.join(dir_0, 'jd_R_ECEF_to_SunFrame_hist.txt'))

dir_1 = os.path.join(OUTPUT_DIR, 'case_5_years_sc_C1', 'data_output')

for i, jd in enumerate(mid_day_jd_array):
    progress_bar(i, len(mid_day_jd_array))

    day_dir = os.path.join(dir_1, 'day_' + str(i))
    jd_day = np.loadtxt(os.path.join(day_dir, 'jd_vec.csv'))
    np.save(os.path.join(dir_0, 'daily_files', 'jd_array_day_'+str(i)), jd_day)
    
    r_sun_ecef_day = np.loadtxt(os.path.join(day_dir, 'r_sun_ecef.csv'))
    r_sun_from_full_file = full_r_sun[np.isin(full_jd_r_sun, jd_day)]
    assert((r_sun_from_full_file==r_sun_ecef_day).all())
    np.save(os.path.join(dir_0, 'daily_files', 'r_Sun_ECEF_day_'+str(i)), r_sun_ecef_day)

    day_R = full_R_ECEF_to_SunFrame_hist[np.isin(rull_jd_R_hist, jd_day), :, :]
    np.save(os.path.join(dir_0, 'daily_files', 'R_ECEF_to_SunFrame_day_'+str(i)), day_R)





KeyboardInterrupt: 